# 05 — Mapping the per-capita crime rate

Every earlier chart showed boroughs as bars or points on an axis — useful,
but London's boroughs have a real spatial layout (which ones are
central vs. outer, which are neighbours) that a bar chart can't show. This
notebook plots the per-capita crime rate from `03_percapita.ipynb` as an
actual map: first a static one, then an interactive one you can hover over.

In [ ]:
import sys
sys.path.append("../../src")

import folium
import matplotlib.pyplot as plt
import seaborn as sns  # importing this registers seaborn's colormaps
                        # (e.g. "rocket") with matplotlib, even though we
                        # don't call seaborn directly in this notebook.

from load_data import load_force_data, load_borough_population, load_lad_boundaries
from clean import clean_crime_data, add_borough_column, LONDON_BOROUGHS

london = load_force_data("london")
london = clean_crime_data(london)
london = add_borough_column(london)
ldn = london[london["Borough"].isin(LONDON_BOROUGHS)]

pop = load_borough_population()
counts = ldn.groupby("Borough").size().rename("Crimes").reset_index()
merged = counts.merge(pop, on="Borough", validate="one_to_one")
merged["rate_per_1000"] = merged["Crimes"] / merged["Population"] * 1000
merged.head()

## Loading borough boundary shapes

So far every DataFrame we've used has held only numbers and text. A
**GeoDataFrame** (from `geopandas`) is the same idea plus one extra column,
`geometry`, holding each row's actual shape (here, a borough's outline as a
polygon) — that's what lets us draw a map instead of a chart.

In [ ]:
boundaries = load_lad_boundaries(LONDON_BOROUGHS)

# A regular merge, exactly like the ones in 03/04 -- geometry is just
# another column here, not a special case.
geo = boundaries.merge(merged, on="Borough", validate="one_to_one", how="left")
geo["rate_per_1000"].isna().sum()  # 0 == every borough's shape matched a row

## Chart — static choropleth map

A choropleth shades each area by a value — the map equivalent of the
heatmap in notebook 02. Same rule applies: one sequential hue (light to
dark), not a rainbow, since this encodes a single continuous magnitude.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
geo.plot(
    column="rate_per_1000",
    cmap="rocket_r",
    linewidth=0.6,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Crimes per 1,000 residents (annual)", "shrink": 0.6},
    ax=ax,
)
ax.set_title("Per-capita crime rate by London borough")
ax.set_axis_off()  # lat/long tick marks aren't meaningful on a small map like this
fig.tight_layout()

Westminster (dark, centre-left of the river bend) and Camden (bright red,
just above it) visually confirm what the numbers said in notebooks 03-04 —
and now you can also see *where* they sit relative to everything else:
both are in inner London, next to each other.

## Chart — interactive map

A static map is fixed; an interactive one lets you hover for exact values
and zoom in. `folium` builds this as a real webpage (it wraps the
JavaScript mapping library Leaflet), so we save it as HTML rather than a
PNG.

In [ ]:
# folium needs a (latitude, longitude) point to center the map on. Our
# coordinates are in EPSG:4326 (plain lat/long degrees) -- a "geographic"
# CRS (coordinate reference system), which distorts distance/area
# calculations like centroid. We temporarily reproject to EPSG:27700
# (British National Grid, measured in metres) to compute an accurate
# centroid, then convert that single point back to lat/long for folium.
projected_centroids = geo.to_crs(epsg=27700).geometry.centroid
centroids = projected_centroids.to_crs(epsg=4326)
map_center = [centroids.y.mean(), centroids.x.mean()]

m = folium.Map(location=map_center, zoom_start=10, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=geo.__geo_interface__,  # __geo_interface__ converts our
                                      # GeoDataFrame to the plain GeoJSON
                                      # dict format folium expects.
    data=geo,
    columns=["Borough", "rate_per_1000"],
    key_on="feature.properties.Borough",  # tells folium which GeoJSON
                                           # field to match against the
                                           # "Borough" column above.
    fill_color="YlOrRd",
    fill_opacity=0.8,
    line_opacity=0.4,
    legend_name="Crimes per 1,000 residents (annual)",
).add_to(m)

# folium.Choropleth alone has no hover tooltip -- layering an invisible
# GeoJson layer on top (fillOpacity 0) adds one without changing how the
# map looks.
folium.GeoJson(
    geo,
    style_function=lambda _: {"fillOpacity": 0, "color": "transparent"},
    tooltip=folium.GeoJsonTooltip(
        fields=["Borough", "rate_per_1000"],
        aliases=["Borough:", "Rate per 1,000:"],
    ),
).add_to(m)

m.save("../../outputs/london_crime_rate_map.html")
m

Running the cell above should show the interactive map directly in the
notebook. It's also saved to `outputs/london_crime_rate_map.html` — open
that file in a browser any time to view or share it without re-running
anything.